# requires-grad-propagation — ex3: chained ops — requires_grad flows through composition

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `requires-grad-propagation`. Running the final beacon cell reports progress against the `Backprop: requires_grad propagation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: requires_grad propagation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`requires-grad-propagation`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "requires-grad-propagation"
DD_SUBTOPIC = "Backprop: requires_grad propagation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## requires_grad propagates THROUGH the graph — composition works

Ex1 gated `requires_grad` on a single op; ex2 extended the scan to kwargs. The deepening move tests COMPOSITION: if op A produces a tensor with `requires_grad=True`, and op B takes that tensor as input, then B's output also has `requires_grad=True` — even if B's OTHER inputs are all `requires_grad=False`.

```python
x = MiniTensor(t.tensor([1.0]), requires_grad=True)
y = MiniTensor(t.tensor([2.0]), requires_grad=False)
z = MiniTensor(t.tensor([3.0]), requires_grad=False)

ab = add(x, y)      # ab.requires_grad == True  (x carries it in)
abc = add(ab, z)    # abc.requires_grad == True (ab carries it forward)
# The 'rg=True' bit FLOWS down the graph through each op.
```

**Why this is the propagation invariant.** The three-gate AND in ex1 decides ONE op's output. Apply that AND at every op in a chain and you get: as long as `grad_tracking_enabled` and `is_differentiable` stay True, the `requires_grad` bit propagates from the FIRST grad-tracked input all the way to the final loss tensor.

**Where the propagation STOPS.** If any op in the chain has `is_differentiable=False` (e.g. `t.equal`, `t.argmax`), the chain snaps — that op's output has `requires_grad=False` regardless of input. Downstream ops see only that detached output.

### Exercise 3 — chained ops — requires_grad flows through composition

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze how `requires_grad` propagates through a chain of ops: applying the three-gate rule at each forward op produces a tensor whose `requires_grad` flag is True iff at least one original input had it AND every op in the chain was differentiable.
> Keywords: requires-grad, composition, graph, propagation
> ```

**KCs targeted:** `requires-grad-propagation`, `graph-propagation-is-transitive`

Implement `ex3_chain_propagate(inputs, op_table)`. Simulates a chain of forward ops and reports the final tensor's `requires_grad`.

Inputs:
- `inputs`: list of `MiniTensor` objects with `.requires_grad` attribute (bool). These are the LEAF inputs at the bottom of the chain.
- `op_table`: list of dicts, each describing one op to apply. Each dict has:
  - `'op_inputs'`: list of indices into the previous step's tensors (where the first step indexes into `inputs`). At each step, those tensors become this op's inputs.
  - `'is_differentiable'`: bool — the op's differentiability flag.
  - `'grad_tracking_enabled'`: bool — the global toggle as of this op.

Behaviour at each step:
- Gather the step's input MiniTensors using `op_inputs` indices.
- Compute the OUTPUT MiniTensor's `requires_grad` via the three-gate AND:
  `grad_tracking_enabled AND is_differentiable AND any(input.requires_grad for input in step_inputs)`.
- The output becomes the SINGLE tensor available to the next step at index 0. (For chain simulation we collapse to one output per step.)

Return: `bool` — the final step's output `requires_grad`.

Edge case: if `op_table` is empty, return `False` (no ops, no output).

In [ ]:
def ex3_chain_propagate(inputs, op_table):
    """Simulate forward chain. Return final tensor requires_grad bool."""
    raise NotImplementedError()


def _test_ex3():
    class MiniTensor:
        def __init__(self, requires_grad=False):
            self.requires_grad = requires_grad

    # === Single op, grad input, all gates True → True ===
    x = MiniTensor(requires_grad=True)
    table = [{'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True}]
    assert ex3_chain_propagate([x], table) is True

    # === Single op, no grad input → False ===
    x = MiniTensor(requires_grad=False)
    table = [{'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True}]
    assert ex3_chain_propagate([x], table) is False

    # === Single op, grad input but toggle off → False ===
    x = MiniTensor(requires_grad=True)
    table = [{'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': False}]
    assert ex3_chain_propagate([x], table) is False

    # === Single op, grad input but not differentiable → False ===
    x = MiniTensor(requires_grad=True)
    table = [{'op_inputs': [0], 'is_differentiable': False, 'grad_tracking_enabled': True}]
    assert ex3_chain_propagate([x], table) is False

    # === Two-op chain: rg propagates from step 1 to step 2 ===
    x = MiniTensor(requires_grad=True)
    table = [
        {'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True},
        {'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True},
    ]
    assert ex3_chain_propagate([x], table) is True

    # === Chain SNAPS at non-differentiable op ===
    x = MiniTensor(requires_grad=True)
    table = [
        {'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True},
        {'op_inputs': [0], 'is_differentiable': False, 'grad_tracking_enabled': True},  # SNAP
        {'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True},
    ]
    assert ex3_chain_propagate([x], table) is False, 'non-diff op must snap the chain'

    # === Mixed inputs: only one needs requires_grad ===
    x = MiniTensor(requires_grad=True)
    y = MiniTensor(requires_grad=False)
    table = [{'op_inputs': [0, 1], 'is_differentiable': True, 'grad_tracking_enabled': True}]
    assert ex3_chain_propagate([x, y], table) is True

    # === All inputs grad-free → False even if differentiable ===
    x = MiniTensor(requires_grad=False)
    y = MiniTensor(requires_grad=False)
    table = [{'op_inputs': [0, 1], 'is_differentiable': True, 'grad_tracking_enabled': True}]
    assert ex3_chain_propagate([x, y], table) is False

    # === Empty op_table → False (no output to flag) ===
    assert ex3_chain_propagate([MiniTensor(requires_grad=True)], []) is False

    # === Long chain: 5 differentiable ops with grad input → still True ===
    x = MiniTensor(requires_grad=True)
    table = [{'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True}] * 5
    assert ex3_chain_propagate([x], table) is True

    # === Toggle off at any step in the chain → False ===
    x = MiniTensor(requires_grad=True)
    table = [
        {'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True},
        {'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': False},  # turn off
        {'op_inputs': [0], 'is_differentiable': True, 'grad_tracking_enabled': True},
    ]
    assert ex3_chain_propagate([x], table) is False, 'toggle-off step must propagate False forward'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_chain_propagate(inputs, op_table):
    if not op_table:
        return False
    # The first step indexes into the original `inputs` list.
    current = list(inputs)
    out = None
    for step in op_table:
        step_inputs = [current[i] for i in step['op_inputs']]
        any_input_rg = any(getattr(a, 'requires_grad', False) for a in step_inputs)
        out_rg = (
            step['grad_tracking_enabled']
            and step['is_differentiable']
            and any_input_rg
        )
        # Build a tiny dummy with the propagated flag.
        out_obj = type('Out', (), {'requires_grad': out_rg})()
        current = [out_obj]  # next step indexes into a 1-tensor list
        out = out_obj
    return out.requires_grad
```

**Three-gate AND at each step is the entire invariant.** No extra state — just apply ex1's rule to the step's inputs, store the result, feed it forward. Composition emerges from iteration; you don't need a graph data structure for this drill.

**Once the chain SNAPS, it stays snapped.** The propagation is ASYMMETRIC: once a step outputs `requires_grad=False`, every downstream step sees that False as its only input and outputs False too (since `any(False) == False`). PyTorch's own autograd matches this — once you `.detach()`, no downstream op can re-attach the gradient.

**Toggle changes are per-op.** The drill's `grad_tracking_enabled` field is per-step because the toggle can flip between ops (entering/exiting a `no_grad()` block). A step's output requires_grad depends on the toggle AT THAT STEP, not at the start of the chain.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()